# Train Test Creator

Reads the pre-selected **`<target>__lb<L>__final`** VIEW from `unified_schema_<ticker>`
and builds windowed train/val/test tensors. Each **sample** is a
`(LOOKBACK_DAY, n_features)` window of all view features ending on day *t*, with
**label = `target` at day *t*** (the 5-day-ahead return `return_5day`).

Feature selection + TA tuning are already done upstream in
`unified_schema_creator.ipynb` (the view is pre-selected), so this notebook only:
**clean → chronological split → scale (train-fit) → window → save**.

## Import Libraries

In [1]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath(".."))

from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import PostgreSQLConnectionDto
from logger.logger import Logger
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from utils.constants import DATABASE_MAIN_V2

load_dotenv()

True

## Parameters

In [2]:
TICKER = "vcb"
TARGET = "return_5day"       # the pool__targets column selected upstream
LOOKBACK_DAY = 5             # window length (must match the __final view's lb)
TARGET_HORIZON = 5           # target looks 5 trading days into the future

SCHEMA = f"unified_schema_{TICKER.lower()}"
VIEW = f"{TARGET}__lb{LOOKBACK_DAY}__final"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = round(1 - TRAIN_RATIO - VAL_RATIO, 4)  # 0.15 (implicit)

DATE_COL = "date"
TARGET_COLUMN = "target"
SCALER_TAG = "std"           # std = StandardScaler
SCALE_TARGET = True          # standardize target (scaler saved for inverse-transform)
RANDOM_STATE = 42

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", 5432))
PG_USER = os.getenv("POSTGRES_USER", "postgres")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

DATASET_NAME = (
    f"{TICKER.lower()}_{TARGET}"
    f"_lb{LOOKBACK_DAY}_h{TARGET_HORIZON}_final"
    f"_tr{int(TRAIN_RATIO * 100)}_val{int(VAL_RATIO * 100)}_test{int(TEST_RATIO * 100)}"
    f"_{SCALER_TAG}"
)
OUTPUT_DIR = os.path.join("../train_test_set", DATASET_NAME)
print(f"Source view : {SCHEMA}.{VIEW}")
print(f"Dataset name: {DATASET_NAME}")

Source view : unified_schema_vcb.return_5day__lb5__final
Dataset name: vcb_return_5day_lb5_h5_final_tr70_val15_test15_std


## Load Data

In [3]:
logger = Logger(file_name="../../logs/train_test_creator")

driver = PostgreSQLDriver(logger=logger)
driver.connect(
    PostgreSQLConnectionDto(
        logger=logger,
        host=PG_HOST,
        user=PG_USER,
        password=PG_PASSWORD,
        port=PG_PORT,
        database=DATABASE_MAIN_V2,
    )
)

df = driver.select(schema_name=SCHEMA, table_name=VIEW, order_by=[DATE_COL])

driver.disconnect()

print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {SCHEMA}.{VIEW}")
df

Loaded 4242 rows, 155 columns from unified_schema_vcb.return_5day__lb5__final


,date,close,foreign_room,own_pct,f_net_vol,f_buy_val,val_matched_bn,volume,f_sell_vol,pct_change,...,natr_14_signal_slope,ht_dcperiod_hist_10_slope,mfi_14_signal_slope,volatility_21,stoch_5_3_3_kd_dist,ht_dcphase_signal_10,macd_12_26_9_hist_abs,ht_dcphase_signal_10_slope,trix_15_hist,target
0,2009-06-30,9132.805,82612532.0,NaN,4100.0,2.460000e+08,17.64,294070.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.058333
1,2009-07-01,9208.911,82441852.0,NaN,170680.0,2.260555e+11,389.79,6248390.0,3420000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.082645
2,2009-07-02,8828.378,82445952.0,NaN,-49670.0,3.280495e+09,88.93,1515670.0,104850.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.068966
3,2009-07-03,8523.951,82445952.0,NaN,-215390.0,4.261395e+09,50.68,899720.0,290680.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.080357
4,2009-07-06,8904.484,82445952.0,NaN,-294930.0,4.348780e+09,90.18,1571740.0,370000.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.162393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-06-22,61300.000,816867042.0,20.22,-617170.0,4.996183e+09,165.05,2686300.0,698484.0,-0.006483,...,-0.042964,0.119630,NaN,0.007758,-16.750841,99.49270,113.68186,-8.972617,-0.084348,NaN
4238,2026-06-23,61300.000,818736677.0,20.20,-148955.0,1.519092e+10,320.49,5176000.0,394290.0,0.000000,...,-0.032694,0.111270,NaN,0.007691,-6.577681,92.31064,116.85823,-7.182062,-0.079497,NaN
4239,2026-06-24,61000.000,818942712.0,20.20,-136600.0,5.317174e+09,155.57,2542500.0,223500.0,-0.004894,...,-0.033926,0.067967,NaN,0.007133,-8.465609,87.04564,132.21176,-5.265000,-0.075509,NaN
4240,2026-06-25,60800.000,819584516.0,20.19,-424339.0,4.151386e+09,188.58,3093500.0,492439.0,-0.003279,...,-0.036485,0.022889,NaN,0.007134,-6.481482,81.75899,147.70973,-5.286652,-0.072463,NaN


## Clean & Classify Columns

In [4]:
# Drop the NaN-target tail (last TARGET_HORIZON rows have an incomplete future)
df_clean = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
dates = pd.to_datetime(df_clean[DATE_COL])
target_series = pd.to_numeric(df_clean[TARGET_COLUMN], errors="coerce")

# Features = every view column except date + target (already selected upstream)
feature_cols = [c for c in df_clean.columns if c not in (DATE_COL, TARGET_COLUMN)]
feature_df = df_clean[feature_cols].apply(pd.to_numeric, errors="coerce")

# Fill NaN (macro leading gaps, TA warmup) — forward then back fill
nan_before = int(feature_df.isna().sum().sum())
feature_df = feature_df.ffill().bfill()
nan_after = int(feature_df.isna().sum().sum())
print(f"Dropped {len(df) - len(df_clean)} NaN-target tail rows; feature NaN {nan_before} -> {nan_after}")

# Bounded columns that should NOT be scaled:
#   - cyclical encodings (sin/cos in [-1, 1])
#   - binary 0/1 flags (calendar flags, TA crossover/band flags)
cyclical_cols = [c for c in feature_df.columns if c.endswith("_sin") or c.endswith("_cos")]
binary_cols = [
    c for c in feature_df.columns
    if set(feature_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]
bounded_cols = sorted(set(cyclical_cols) | set(binary_cols))
scale_cols = [c for c in feature_df.columns if c not in set(bounded_cols)]

print(f"Feature matrix: {feature_df.shape}  |  bounded (not scaled): {len(bounded_cols)}"
      f"  |  continuous (scaled): {len(scale_cols)}")
feature_df

Dropped 5 NaN-target tail rows; feature NaN 77729 -> 0
Feature matrix: (4237, 153)  |  bounded (not scaled): 10  |  continuous (scaled): 143


,close,foreign_room,own_pct,f_net_vol,f_buy_val,val_matched_bn,volume,f_sell_vol,pct_change,vol_negotiated,...,close_wma_50_100_dist_slope,natr_14_signal_slope,ht_dcperiod_hist_10_slope,mfi_14_signal_slope,volatility_21,stoch_5_3_3_kd_dist,ht_dcphase_signal_10,macd_12_26_9_hist_abs,ht_dcphase_signal_10_slope,trix_15_hist
0,9132.805,82612532.0,0.00,4100.0,2.460000e+08,17.64,294070.0,0.0,-0.008674,0.0,...,-11.330280,-0.027436,1.280511,0.802234,0.033527,-4.444445,80.12363,97.526690,2.724777,-0.150984
1,9208.911,82441852.0,0.00,170680.0,2.260555e+11,389.79,6248390.0,3420000.0,-0.008674,3.0,...,-11.330280,-0.027436,1.280511,0.802234,0.033527,-4.444445,80.12363,97.526690,2.724777,-0.150984
2,8828.378,82445952.0,0.00,-49670.0,3.280495e+09,88.93,1515670.0,104850.0,-0.008674,0.0,...,-11.330280,-0.027436,1.280511,0.802234,0.033527,-4.444445,80.12363,97.526690,2.724777,-0.150984
3,8523.951,82445952.0,0.00,-215390.0,4.261395e+09,50.68,899720.0,290680.0,-0.008674,0.0,...,-11.330280,-0.027436,1.280511,0.802234,0.033527,-4.444445,80.12363,97.526690,2.724777,-0.150984
4,8904.484,82445952.0,0.00,-294930.0,4.348780e+09,90.18,1571740.0,370000.0,-0.008674,0.0,...,-11.330280,-0.027436,1.280511,0.802234,0.033527,-4.444445,80.12363,97.526690,2.724777,-0.150984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4232,61600.000,811618973.0,20.28,-1517391.0,2.228014e+10,323.38,5222300.0,1877199.0,0.000000,0.0,...,49.678510,-0.045923,0.058122,2.762535,0.013325,0.228938,189.65546,206.498170,-37.168650,-0.096927
4233,61800.000,811732555.0,20.28,-714066.0,2.858882e+10,192.49,3110000.0,1175966.0,0.003247,0.0,...,48.670550,-0.047361,0.073598,2.762535,0.009796,-0.414863,160.99728,169.506150,-28.658178,-0.099765
4234,62200.000,812111195.0,20.28,-2889388.0,1.270252e+10,423.45,6825500.0,3094137.0,0.006472,0.0,...,51.612503,-0.047291,0.061260,2.762535,0.008869,6.405400,138.36197,113.518470,-22.635315,-0.097857
4235,61600.000,813459783.0,20.26,-2048756.0,2.098636e+10,287.74,4650700.0,2387956.0,-0.009646,0.0,...,34.296253,-0.043134,0.075733,2.762535,0.008965,-6.415806,121.13626,112.118576,-17.225706,-0.094313


## Split Indices (chronological)

In [5]:
n_rows = len(feature_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))

print(f"Total rows         : {n_rows}")
print(f"Train rows [0:{train_end}]   {dates.iloc[0].date()} -> {dates.iloc[train_end - 1].date()}")
print(f"Val   rows [{train_end}:{val_end}]   {dates.iloc[train_end].date()} -> {dates.iloc[val_end - 1].date()}")
print(f"Test  rows [{val_end}:{n_rows}]   {dates.iloc[val_end].date()} -> {dates.iloc[n_rows - 1].date()}")

Total rows         : 4237
Train rows [0:2965]   2009-06-30 -> 2021-05-14
Val   rows [2965:3601]   2021-05-17 -> 2023-11-24
Test  rows [3601:4237]   2023-11-27 -> 2026-06-19


## Normalize (fit on train rows only)

In [6]:
# Order columns [scaled continuous ... | bounded], then scale (train-fit only)
ordered_cols = scale_cols + bounded_cols
feat = feature_df[ordered_cols].copy()

# --- Feature scaler: fit on TRAIN rows only, transform the whole array ---
feature_scaler = StandardScaler()
feature_scaler.fit(feat.iloc[:train_end][scale_cols].values)
feat[scale_cols] = feature_scaler.transform(feat[scale_cols].values)

# --- Target: optionally standardize (fit on TRAIN rows only) ---
if SCALE_TARGET:
    target_scaler = StandardScaler()
    target_scaler.fit(target_series.iloc[:train_end].values.reshape(-1, 1))
    y_full = target_scaler.transform(target_series.values.reshape(-1, 1)).ravel()
else:
    target_scaler = None
    y_full = target_series.values

X_arr = feat.values.astype(np.float32)
y_arr = y_full.astype(np.float32)

print(f"Scaled feature array: {X_arr.shape}  ({len(scale_cols)} scaled, {len(bounded_cols)} bounded)")
if SCALE_TARGET:
    print(f"Target scaled  mean~0: {y_arr[:train_end].mean():.4f}  std~1: {y_arr[:train_end].std():.4f}")

Scaled feature array: (4237, 153)  (143 scaled, 10 bounded)
Target scaled  mean~0: 0.0000  std~1: 1.0000


## Create Windows (3D tensors)

In [7]:
def make_windows(X, y, start, end):
    """Sliding windows of length LOOKBACK_DAY; target is the value at the window's last day.

    Each split starts LOOKBACK_DAY-1 rows early so its first window is complete
    without borrowing target rows from the previous split (no leakage —
    features are scaled with train-only statistics, targets only look forward).
    """
    xs, ys = [], []
    for i in range(start, end - LOOKBACK_DAY + 1):
        xs.append(X[i : i + LOOKBACK_DAY])
        ys.append(y[i + LOOKBACK_DAY - 1])
    return np.stack(xs).astype(np.float32), np.array(ys, dtype=np.float32)


# Label date per window = date of the window's last day (i + LOOKBACK_DAY - 1)
date_arr = dates.dt.strftime("%Y-%m-%d").to_numpy()
def window_dates(start, end):
    return np.array([date_arr[i + LOOKBACK_DAY - 1]
                     for i in range(start, end - LOOKBACK_DAY + 1)])

X_train, y_train = make_windows(X_arr, y_arr, 0, train_end)
X_val,   y_val   = make_windows(X_arr, y_arr, train_end - (LOOKBACK_DAY - 1), val_end)
X_test,  y_test  = make_windows(X_arr, y_arr, val_end - (LOOKBACK_DAY - 1), n_rows)

dates_train = window_dates(0, train_end)
dates_val   = window_dates(train_end - (LOOKBACK_DAY - 1), val_end)
dates_test  = window_dates(val_end - (LOOKBACK_DAY - 1), n_rows)

print(f"X_train : {X_train.shape}  y_train : {y_train.shape}  dates_train : {dates_train.shape}")
print(f"X_val   : {X_val.shape}  y_val   : {y_val.shape}  dates_val : {dates_val.shape}")
print(f"X_test  : {X_test.shape}  y_test  : {y_test.shape}  dates_test : {dates_test.shape}")

X_train : (2961, 5, 153)  y_train : (2961,)  dates_train : (2961,)
X_val   : (636, 5, 153)  y_val   : (636,)  dates_val : (636,)
X_test  : (636, 5, 153)  y_test  : (636,)  dates_test : (636,)


## Save Model-Ready Datasets

In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tensors
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Per-window label dates (the window's last day) — used for plots/backtests
np.save(os.path.join(OUTPUT_DIR, "dates_train.npy"), dates_train)
np.save(os.path.join(OUTPUT_DIR, "dates_val.npy"), dates_val)
np.save(os.path.join(OUTPUT_DIR, "dates_test.npy"), dates_test)

# Scalers (joblib handles sklearn objects well)
joblib.dump(feature_scaler, os.path.join(OUTPUT_DIR, "feature_scaler.pkl"))
if target_scaler is not None:
    joblib.dump(target_scaler, os.path.join(OUTPUT_DIR, "target_scaler.pkl"))

# Metadata
metadata = {
    "dataset_name": DATASET_NAME,
    "ticker": TICKER.lower(),
    "schema": SCHEMA,
    "source_view": VIEW,
    "lookback_day": LOOKBACK_DAY,
    "target_horizon": TARGET_HORIZON,
    "target": TARGET,
    "target_column": TARGET_COLUMN,
    "n_features": len(ordered_cols),
    "feature_columns": ordered_cols,
    "scaled_columns": scale_cols,
    "bounded_columns": bounded_cols,
    "feature_selection": (
        "done upstream in unified_schema_creator.ipynb; the "
        f"{VIEW} view is already the selected feature set"
    ),
    "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "shapes": {
        "X_train": list(X_train.shape), "y_train": list(y_train.shape),
        "X_val": list(X_val.shape),     "y_val": list(y_val.shape),
        "X_test": list(X_test.shape),   "y_test": list(y_test.shape),
    },
    "date_ranges": {
        "train": [str(dates.iloc[0].date()), str(dates.iloc[train_end - 1].date())],
        "val":   [str(dates.iloc[train_end].date()), str(dates.iloc[val_end - 1].date())],
        "test":  [str(dates.iloc[val_end].date()), str(dates.iloc[n_rows - 1].date())],
    },
    "scaler": {
        "feature": "StandardScaler",
        "target": "StandardScaler" if SCALE_TARGET else None,
    },
    "nan_handling": {"target": "dropped tail NaN rows", "features": "ffill+bfill"},
    "random_state": RANDOM_STATE,
    "created_at": pd.Timestamp.now().strftime("%Y-%m-%d"),
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:24s} {size:8.2f} MB")

Saved to ../train_test_set\vcb_return_5day_lb5_h5_final_tr70_val15_test15_std
  X_test.npy                   1.95 MB
  X_train.npy                  9.06 MB
  X_val.npy                    1.95 MB
  dates_test.npy               0.03 MB
  dates_train.npy              0.12 MB
  dates_val.npy                0.03 MB
  feature_scaler.pkl           0.00 MB
  metadata.json                0.01 MB
  target_scaler.pkl            0.00 MB
  y_test.npy                   0.00 MB
  y_train.npy                  0.01 MB
  y_val.npy                    0.00 MB
